In [1]:
import xml.etree.ElementTree as ET

def parse_cte(xml_source):
    if xml_source.strip().startswith('<'):
        root = ET.fromstring(xml_source)
    else:
        root = ET.parse(xml_source).getroot()

    ns = {'cte': 'http://www.portalfiscal.inf.br/cte'}

    def txt(node, path):
        if node is None:
            return None
        el = node.find(path, ns)
        return el.text.strip() if el is not None and el.text else None

    def flt(node, path):
        v = txt(node, path)
        return float(v) if v else 0.0

    ide    = root.find('.//cte:ide', ns)
    prot   = root.find('.//cte:infProt', ns)
    emit   = root.find('.//cte:emit', ns)
    rem    = root.find('.//cte:rem', ns)
    dest   = root.find('.//cte:dest', ns)
    icms00 = root.find('.//cte:ICMS00', ns)
    ibscbs = root.find('.//cte:gIBSCBS', ns)
    ibsuf  = root.find('.//cte:gIBSUF', ns)
    cbs_no = root.find('.//cte:gCBS', ns)
    vprest = root.find('.//cte:vPrest', ns)
    carga  = root.find('.//cte:infCarga', ns)

    cfop = txt(ide, 'cte:CFOP')
    cfop_map = {
        '6353': 'Venda / Uso Comercial (interestadual)',
        '5353': 'Venda / Uso Comercial (intraestadual)',
        '6352': 'Transferência (interestadual)',
        '5352': 'Transferência (intraestadual)',
        '6949': 'Assistência Técnica / Outros (interestadual)',
        '5949': 'Assistência Técnica / Outros (intraestadual)',
    }

    componentes = {}
    if vprest is not None:
        for comp in vprest.findall('cte:Comp', ns):
            nome  = txt(comp, 'cte:xNome')
            valor = txt(comp, 'cte:vComp')
            if nome:
                componentes[nome] = float(valor) if valor else 0.0

    # Bases e alíquotas
    bc_icms    = flt(icms00, 'cte:vBC')
    aliq_icms  = flt(icms00, 'cte:pICMS')
    vl_icms    = flt(icms00, 'cte:vICMS')

    bc_ibs     = flt(ibscbs, 'cte:vBC')
    aliq_ibs   = flt(ibsuf,  'cte:pIBSUF')
    vl_ibs     = flt(ibsuf,  'cte:vIBSUF')
    aliq_cbs   = flt(cbs_no, 'cte:pCBS')
    vl_cbs     = flt(cbs_no, 'cte:vCBS')

    # Validação: alíquota × BC deve bater com o valor declarado
    def valida(bc, aliq_pct, declarado):
        calculado = round(bc * aliq_pct / 100, 2)
        ok = calculado == round(declarado, 2)
        return calculado, '✅' if ok else '❌'

    icms_calc,  icms_ok  = valida(bc_icms, aliq_icms, vl_icms)
    ibs_calc,   ibs_ok   = valida(bc_ibs,  aliq_ibs,  vl_ibs)
    cbs_calc,   cbs_ok   = valida(bc_ibs,  aliq_cbs,  vl_cbs)

    dados = {
        # ── Identificação ──────────────────────────────────────────
        'chave_acesso':      txt(prot, 'cte:chCTe')   if prot is not None else None,
        'numero_cte':        txt(ide,  'cte:nCT'),
        'serie':             txt(ide,  'cte:serie'),
        'data_emissao':      txt(ide,  'cte:dhEmi'),
        'cfop':              cfop,
        'nat_operacao':      txt(ide,  'cte:natOp'),
        'finalidade':        cfop_map.get(cfop, f'Verificar CFOP {cfop}'),
        'origem':            txt(ide,  'cte:xMunIni'),
        'uf_origem':         txt(ide,  'cte:UFIni'),
        'destino':           txt(ide,  'cte:xMunFim'),
        'uf_destino':        txt(ide,  'cte:UFFim'),

        # ── Partes ─────────────────────────────────────────────────
        'emitente':          txt(emit, 'cte:xNome'),
        'cnpj_emitente':     txt(emit, 'cte:CNPJ'),
        'remetente':         txt(rem,  'cte:xNome'),
        'cnpj_remetente':    txt(rem,  'cte:CNPJ'),
        'destinatario':      txt(dest, 'cte:xNome'),
        'cnpj_destinatario': txt(dest, 'cte:CNPJ'),

        # ── Carga ──────────────────────────────────────────────────
        'produto':           txt(carga, 'cte:proPred'),
        'valor_carga':       flt(carga, 'cte:vCarga'),

        # ── Frete ──────────────────────────────────────────────────
        'valor_total_frete': flt(vprest, 'cte:vTPrest'),
        'componentes_frete': componentes,

        # ── Impostos e bases ───────────────────────────────────────
        'impostos': {
            'ICMS': {
                'base_calculo': bc_icms,
                'aliquota_pct': aliq_icms,
                'valor_declarado': vl_icms,
                'valor_calculado': icms_calc,
                'consistencia': icms_ok,
            },
            'IBS_UF': {
                'base_calculo': bc_ibs,
                'aliquota_pct': aliq_ibs,
                'valor_declarado': vl_ibs,
                'valor_calculado': ibs_calc,
                'consistencia': ibs_ok,
            },
            'CBS': {
                'base_calculo': bc_ibs,   # mesma BC do IBS
                'aliquota_pct': aliq_cbs,
                'valor_declarado': vl_cbs,
                'valor_calculado': cbs_calc,
                'consistencia': cbs_ok,
            },
        },

        # ── Protocolo ──────────────────────────────────────────────
        'protocolo':  txt(prot, 'cte:nProt')   if prot is not None else None,
        'status':     txt(prot, 'cte:xMotivo') if prot is not None else None,
        'nfe_ref':    txt(root.find('.//cte:infNFe', ns), 'cte:chave'),
    }

    return dados


# ── Exibição ────────────────────────────────────────────────────────
def exibir_cte(dados):
    print("=" * 55)
    print("  CT-e Nº", dados['numero_cte'], "| Série", dados['serie'])
    print("=" * 55)

    print("\n📋 IDENTIFICAÇÃO")
    for k in ['chave_acesso','data_emissao','cfop','nat_operacao','finalidade']:
        print(f"  {k:<22}: {dados[k]}")

    print("\n🗺️  ROTA")
    print(f"  {'origem':<22}: {dados['origem']} / {dados['uf_origem']}")
    print(f"  {'destino':<22}: {dados['destino']} / {dados['uf_destino']}")

    print("\n🏢 PARTES")
    for k in ['emitente','cnpj_emitente','remetente','cnpj_remetente','destinatario','cnpj_destinatario']:
        print(f"  {k:<22}: {dados[k]}")

    print("\n📦 CARGA")
    print(f"  {'produto':<22}: {dados['produto']}")
    print(f"  {'valor_carga':<22}: R$ {dados['valor_carga']:.2f}")

    print("\n💰 FRETE")
    print(f"  {'total':<22}: R$ {dados['valor_total_frete']:.2f}")
    for nome, valor in dados['componentes_frete'].items():
        print(f"    {nome:<20}: R$ {valor:.2f}")

    print("\n🧾 IMPOSTOS")
    print(f"  {'Imposto':<8} {'BC':>10} {'Alíq%':>7} {'Declarado':>11} {'Calculado':>11} {'OK?':>5}")
    print(f"  {'-'*8} {'-'*10} {'-'*7} {'-'*11} {'-'*11} {'-'*5}")
    for imp, v in dados['impostos'].items():
        print(f"  {imp:<8} {v['base_calculo']:>10.2f} {v['aliquota_pct']:>7.2f} "
              f"{v['valor_declarado']:>11.2f} {v['valor_calculado']:>11.2f} {v['consistencia']:>5}")

    print("\n✅ PROTOCOLO")
    for k in ['protocolo','status','nfe_ref']:
        print(f"  {k:<22}: {dados[k]}")
    print("=" * 55)


dados = parse_cte(r'C:\Users\tiago.premiano\Desktop\MAIARA-VALTER\41260406259224000107570020000047291000981336-cte.xml' )    
exibir_cte(dados)

  CT-e Nº 4729 | Série 2

📋 IDENTIFICAÇÃO
  chave_acesso          : 41260406259224000107570020000047291000981336
  data_emissao          : 2026-04-17T16:19:37-03:00
  cfop                  : 6353
  nat_operacao          : Transp a est comercial
  finalidade            : Venda / Uso Comercial (interestadual)

🗺️  ROTA
  origem                : SARANDI / PR
  destino               : JUNDIAI / SP

🏢 PARTES
  emitente              : VPX LOGISTICA E TRANSPORTES LTDA. - 008
  cnpj_emitente         : 06259224000107
  remetente             : GRUPO SOHOME LTDA
  cnpj_remetente        : 02749435000169
  destinatario          : WESTWING COMERCIO VAREJISTA S.A.
  cnpj_destinatario     : 14776142000907

📦 CARGA
  produto               : MODELO ODISSEIA
  valor_carga           : R$ 1736.54

💰 FRETE
  total                 : R$ 169.89
    FRETE VALOR         : R$ 138.92
    DESPACHO            : R$ 3.95
    PEDAGIO             : R$ 12.00
    ADICIONAL FRETE     : R$ 15.02

🧾 IMPOSTOS
  Imposto       

In [2]:
dados

{'chave_acesso': '41260406259224000107570020000047291000981336',
 'numero_cte': '4729',
 'serie': '2',
 'data_emissao': '2026-04-17T16:19:37-03:00',
 'cfop': '6353',
 'nat_operacao': 'Transp a est comercial',
 'finalidade': 'Venda / Uso Comercial (interestadual)',
 'origem': 'SARANDI',
 'uf_origem': 'PR',
 'destino': 'JUNDIAI',
 'uf_destino': 'SP',
 'emitente': 'VPX LOGISTICA E TRANSPORTES LTDA. - 008',
 'cnpj_emitente': '06259224000107',
 'remetente': 'GRUPO SOHOME LTDA',
 'cnpj_remetente': '02749435000169',
 'destinatario': 'WESTWING COMERCIO VAREJISTA S.A.',
 'cnpj_destinatario': '14776142000907',
 'produto': 'MODELO ODISSEIA',
 'valor_carga': 1736.54,
 'valor_total_frete': 169.89,
 'componentes_frete': {'FRETE VALOR': 138.92,
  'DESPACHO': 3.95,
  'PEDAGIO': 12.0,
  'ADICIONAL FRETE': 15.02},
 'impostos': {'ICMS': {'base_calculo': 169.89,
   'aliquota_pct': 12.0,
   'valor_declarado': 20.39,
   'valor_calculado': 20.39,
   'consistencia': '✅'},
  'IBS_UF': {'base_calculo': 144.04,
